# A linear classifier · Module 07
Python 3 and NumPy only. You will train with the gradient descent from Module 08. The task is three classes of small synthetic images, with a nearest-centroid baseline. Keep the test set out of every decision until the procedure is fixed.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(207)
def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
def cross_entropy(z, y):
    shifted = z - z.max(axis=1, keepdims=True)
    return np.mean(np.log(np.exp(shifted).sum(axis=1)) - shifted[np.arange(len(y)), y])
with np.errstate(over='ignore'): print('exp(1000) alone overflows to', np.exp(1000.))
print('the stable softmax subtracts the largest score first:', softmax(np.array([[1000., 1001.]])))
assert np.allclose(softmax(np.array([[1000., 1001.]])), softmax(np.array([[0., 1.]])))
scores = np.array([[2., 1., 0.]])
print('softmax:', softmax(scores), ' shifted by 5:', softmax(scores + 5))
assert np.allclose(softmax(scores), softmax(scores + 5))
print('loss when the true class gets 0.8:', -np.log(.8), ' when it gets 0.01:', -np.log(.01))
print('Gaussian residuals give squared error: NLL = (y-f)^2/(2 s^2) + constant, so minimizing one minimizes the other.')


exp(1000) alone overflows to inf
the stable softmax subtracts the largest score first: [[0.2689 0.7311]]
softmax: [[0.6652 0.2447 0.09  ]]  shifted by 5: [[0.6652 0.2447 0.09  ]]
loss when the true class gets 0.8: 0.2231435513142097  when it gets 0.01: 4.605170185988091
Gaussian residuals give squared error: NLL = (y-f)^2/(2 s^2) + constant, so minimizing one minimizes the other.


## Part 2: Data and a protocol, declared first
Three classes: horizontal bar, vertical bar and diagonal. The split is fixed once. Standardization statistics come from training data only. The comparison is nearest centroid against softmax regression, and test accuracy is read once at the end.


In [2]:
CLASSES = ['horizontal', 'vertical', 'diagonal']
def make_images(n, noise=.7):
    y = rng.integers(0, 3, n); images = np.zeros((n, 8, 8))
    for i, label in enumerate(y):
        p = int(rng.integers(1, 6))
        if label == 0: images[i, p:p+2, :] = 1.
        elif label == 1: images[i, :, p:p+2] = 1.
        else: images[i, np.arange(8), (np.arange(8) + p) % 8] = 1.5
    return (images + rng.normal(0, noise, images.shape)).reshape(n, -1), y
X, y = make_images(600)
Xtr, Xval, Xtest = X[:360], X[360:480], X[480:]; ytr, yval, ytest = y[:360], y[360:480], y[480:]
mu, scale = Xtr.mean(axis=0), Xtr.std(axis=0)
Str, Sval, Stest = (Xtr - mu) / scale, (Xval - mu) / scale, (Xtest - mu) / scale
centers = np.stack([Str[ytr == c].mean(axis=0) for c in range(3)])
def centroid_predict(S): return np.sum((S[:, None, :] - centers[None, :, :]) ** 2, axis=2).argmin(axis=1)
print('sizes:', Str.shape, Sval.shape, Stest.shape)
print('nearest-centroid validation accuracy:', np.mean(centroid_predict(Sval) == yval))


sizes: (360, 64) (120, 64) (120, 64)
nearest-centroid validation accuracy: 0.775


## Part 3: Softmax regression, gradient checked before training
With row-stored examples, `logits = S @ W + b`, and the gradient comes from (P - Y). A check first, training second. Read the learning curve as you did in Module 08.


In [3]:
def logits(S, W, b): return S @ W + b
def loss_grad(S, y, W, b, weight_decay=0.):
    z = logits(S, W, b); G = (softmax(z) - np.eye(3)[y]) / len(y)
    return cross_entropy(z, y) + .5 * weight_decay * np.sum(W ** 2), S.T @ G + weight_decay * W, G.sum(axis=0)
W = rng.normal(0, .01, (64, 3)); b = np.zeros(3)
_, dW, db = loss_grad(Str[:8], ytr[:8], W, b, .1)
for index in [(0, 0), (17, 2), (40, 1)]:
    plus = W.copy(); plus[index] += 1e-5; minus = W.copy(); minus[index] -= 1e-5
    numeric = (loss_grad(Str[:8], ytr[:8], plus, b, .1)[0] - loss_grad(Str[:8], ytr[:8], minus, b, .1)[0]) / 2e-5
    print('gradient check at', index, 'error', abs(numeric - dW[index]))
    assert abs(numeric - dW[index]) < 1e-7
def train(weight_decay=0., lr=.1, steps=400):
    W = np.zeros((64, 3)); b = np.zeros(3); history = []
    for step in range(steps):
        loss, dW, db = loss_grad(Str, ytr, W, b, weight_decay)
        W -= lr * dW; b -= lr * db
        if step % 100 == 0: history.append(round(float(loss), 4))
    return W, b, history
W, b, history = train()
print('training loss every 100 steps:', history)
assert history[-1] < history[0]
print('softmax-regression validation accuracy:', np.mean(logits(Sval, W, b).argmax(axis=1) == yval))


gradient check at (0, 0) error 1.4306777984529617e-11
gradient check at (17, 2) error 2.2272739208517578e-12
gradient check at (40, 1) error 2.5126428715438465e-12
training loss every 100 steps: [1.0986, 0.3185, 0.2606, 0.2343]
softmax-regression validation accuracy: 0.8166666666666667


## Part 4: Choose a penalty on validation data
A squared penalty is the Gaussian prior of Lesson 26. Compare a few values and choose by validation loss, which sees confidence as well as correctness.


In [4]:
choices = [0., 1e-3, 1e-2, 1e-1, 1.]; scores = []
for penalty in choices:
    Wp, bp, _ = train(penalty)
    scores.append(cross_entropy(logits(Sval, Wp, bp), yval))
    print(f'penalty {penalty:<6} validation NLL {scores[-1]:.4f}  accuracy {np.mean(logits(Sval, Wp, bp).argmax(axis=1) == yval):.3f}  weight norm {np.linalg.norm(Wp):.2f}')
best = choices[int(np.argmin(scores))]
print('selected penalty:', best)
W, b, _ = train(best)


penalty 0.0    validation NLL 0.6201  accuracy 0.817  weight norm 3.86
penalty 0.001  validation NLL 0.6156  accuracy 0.817  weight norm 3.79
penalty 0.01   validation NLL 0.5850  accuracy 0.808  weight norm 3.28
penalty 0.1    validation NLL 0.5679  accuracy 0.800  weight norm 1.68
penalty 1.0    validation NLL 0.8178  accuracy 0.800  weight norm 0.52
selected penalty: 0.1


## Part 5: Test once, then look at the errors
The procedure is now fixed. A confusion matrix shows which classes are mixed up, which a single accuracy hides. Confidence is not correctness, so compare the two by bins: this is a calibration check.


In [5]:
z = logits(Stest, W, b); predicted = z.argmax(axis=1)
print('test count:', len(ytest))
print('nearest-centroid test accuracy:', np.mean(centroid_predict(Stest) == ytest))
print('softmax test accuracy:', np.mean(predicted == ytest), ' test NLL:', cross_entropy(z, ytest))
confusion = np.zeros((3, 3), dtype=int); np.add.at(confusion, (ytest, predicted), 1)
print('rows = true, columns = predicted', CLASSES); print(confusion)
assert confusion.sum() == len(ytest)
confidence = softmax(z).max(axis=1); correct = predicted == ytest
for low, high in [(0., .6), (.6, .8), (.8, .95), (.95, 1.01)]:
    members = (confidence >= low) & (confidence < high)
    if members.any():
        print(f'confidence {low:.2f}-{min(high, 1):.2f}: n={members.sum():3d}  mean confidence {confidence[members].mean():.3f}  accuracy {correct[members].mean():.3f}')
wrong = np.where(~correct)[0]
if len(wrong):
    worst = wrong[confidence[wrong].argmax()]
    print('most confident mistake: true', CLASSES[ytest[worst]], 'predicted', CLASSES[predicted[worst]], 'confidence', confidence[worst].round(3))


test count: 120
nearest-centroid test accuracy: 0.7916666666666666
softmax test accuracy: 0.825  test NLL: 0.5408172327057735
rows = true, columns = predicted ['horizontal', 'vertical', 'diagonal']
[[34  3  4]
 [ 5 31  4]
 [ 3  2 34]]
confidence 0.00-0.60: n= 31  mean confidence 0.478  accuracy 0.613
confidence 0.60-0.80: n= 49  mean confidence 0.701  accuracy 0.837
confidence 0.80-0.95: n= 33  mean confidence 0.884  accuracy 0.970
confidence 0.95-1.00: n=  7  mean confidence 0.969  accuracy 1.000
most confident mistake: true diagonal predicted horizontal confidence 0.95


## Your experiment
Spot two ways leakage could enter this notebook, and show the effect of one of them. Then change one thing, such as adding a fourth class or raising the noise, and explain the change with the confusion matrix. Use a fresh test set for a revised procedure. Submit the baseline, losses, confusion matrix and your split protocol.
